In [0]:
# HIGH_GARDEN_CATALOG_PARAMETER
dbutils.widgets.text("catalog", "high_garden")
catalog = dbutils.widgets.get("catalog").strip() or "high_garden"
print(f"Using Unity Catalog: {catalog}")


In [0]:
import numpy as np
import pandas as pd
import mlflow

from mlflow import MlflowClient
from mlflow.models import infer_signature

In [0]:
mlflow.set_registry_uri(
    "databricks-uc"
)

MODEL_NAME = (
    f"{catalog}.ml.coffee_forecaster"
)

EXPERIMENT_NAME = (
    "/Shared/high-garden-coffee"
)

mlflow.set_experiment(
    EXPERIMENT_NAME
)

In [0]:
metrics_row = (
    spark.table(
        f"{catalog}.gold.model_metrics"
    )
    .filter(
        "model = 'naive'"
    )
    .first()
)

naive_metrics = {
    "mae": float(metrics_row["mae"]),
    "rmse": float(metrics_row["rmse"]),
    "wape": float(metrics_row["wape"]),
    "mase": float(metrics_row["mase"]),
}

print(naive_metrics)

In [0]:
class NaiveForecastModel(
    mlflow.pyfunc.PythonModel
):

    def predict(
        self,
        context,
        model_input,
        params=None
    ):

        predictions = (
            pd.to_numeric(
                model_input["lag_1"],
                errors="coerce"
            )
            .to_numpy(
                dtype=float
            )
        )

        predictions = np.clip(
            predictions,
            a_min=0,
            a_max=None
        )

        return predictions

In [0]:
input_example = pd.DataFrame(
    {
        "lag_1": [
            100000.0,
            250000.0,
            500000.0
        ]
    }
)

In [0]:
test_model = NaiveForecastModel()

example_prediction = (
    test_model.predict(
        None,
        input_example
    )
)

print(
    example_prediction
)

In [0]:
signature = infer_signature(
    input_example,
    example_prediction
)

In [0]:
with mlflow.start_run(
    run_name="naive_champion"
) as run:

    mlflow.log_param(
        "model_type",
        "naive_persistence"
    )

    mlflow.log_param(
        "forecast_horizon",
        "one_step_ahead"
    )

    mlflow.log_param(
        "validation",
        "rolling_origin"
    )

    mlflow.log_param(
        "domain_constraint",
        "non_negative"
    )

    for metric, value in (
        naive_metrics.items()
    ):

        mlflow.log_metric(
            metric,
            value
        )

    model_info = (
        mlflow.pyfunc.log_model(
            name="model",
            python_model=
                NaiveForecastModel(),
            input_example=
                input_example,
            signature=
                signature
        )
    )

    champion_run_id = (
        run.info.run_id
    )

print(
    "Run ID:",
    champion_run_id
)

print(
    "Model URI:",
    model_info.model_uri
)

In [0]:
registered_model = (
    mlflow.register_model(
        model_uri=
            model_info.model_uri,
        name=
            MODEL_NAME
    )
)

In [0]:
print(
    "Registered model:",
    registered_model.name
)

print(
    "Version:",
    registered_model.version
)

In [0]:
client = MlflowClient()

client.set_registered_model_alias(
    MODEL_NAME,
    "Champion",
    registered_model.version
)

In [0]:
champion_info = (
    client
    .get_model_version_by_alias(
        MODEL_NAME,
        "Champion"
    )
)

print(
    "Champion version:",
    champion_info.version
)

In [0]:
CHAMPION_URI = (
    f"models:/{MODEL_NAME}@Champion"
)

champion_model = (
    mlflow.pyfunc.load_model(
        CHAMPION_URI
    )
)

In [0]:
test_input = pd.DataFrame(
    {
        "lag_1": [
            750000.0,
            1250000.0,
            50000.0
        ]
    }
)

test_predictions = (
    champion_model.predict(
        test_input
    )
)

print(
    test_predictions
)

In [0]:
assert (
    np.all(
        test_predictions >= 0
    )
), "Negative predictions detected"

assert (
    len(test_predictions)
    == len(test_input)
), "Prediction count mismatch"

print(
    "Champion model validation passed."
)